In [ ]:
import sys
!{sys.executable} -m pip install pydantic_settings requests

In [ ]:
# imports
import io, sys
import requests
import logging
import json
import datetime
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, field_validator, Field
from typing import Optional

In [ ]:
# Configure logging to show debug messages in the notebook
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# define tools
def exit(reason: str):
    print(f"Exited with: {reason}")

def clock():
    return datetime.datetime

In [ ]:
# create agent settings
class AgentSettings(BaseSettings):
    """
    AgentSettings contains the agent process specific settings 
    """
    chat_api_url: str
    chat_api_key: str
    # optional default chat model name (e.g. gpt-4, gpt-3.5-turbo, etc.)
    chat_api_model: Optional[str] = None

    model_config = ConfigDict(env_file=".env")

    @field_validator("*", mode="before")
    @classmethod
    def empty_str_to_none(cls, v):
        # Empty string is not 'fine' so we convert it to None to make it fail validation
        return None if v == "" else v

In [ ]:
class OpenAIClient():
    """
    The OpenAIClient is an implementation of the LLM_API interface for the OpenAI API-compatible LLM service - e.g. OpenAI itself, 
    DeepSeek, OpenRouter etc.
    """

    def __init__(self, svc_endpoint: str, api_key: str, system_prompt: Optional[str] = None,
                 temperature: Optional[float] = None):
        """
        Initialize the OpenAI-compatible client.
        Args:
            svc_endpoint (str): The service endpoint URL.
            api_key (str): The API key for authentication.
            system_prompt (Optional[str]): An optional system prompt to guide the conversation.
            temperature (Optional[float]): The temperature setting for response variability.
        """
        self.svc_endpoint = svc_endpoint
        self.api_key = api_key
        self.system_prompt = system_prompt
        self.temperature = temperature if temperature is not None else 0.7

    def execute_script(self, python_code):
        #print(f"Executing script: {python_code}")
        buf = io.StringIO()
        old = sys.stdout
        sys.stdout = buf
        try:
            exec(python_code)
        finally:
            sys.stdout = old
        captured = buf.getvalue()
        prefixed = "\n".join("> " + line for line in captured.splitlines())
        print(prefixed)
        
    def chat(self, prompt: str, model: str = None) -> str:
        """
        Send a chat message to the OpenAI API and return the response.
        Args:
            prompt (str): The chat message to send.
            model (str): Optional model name to use for this request.
        Returns:
            str: The response from the LLM API service.
        """

        headers = {
            'Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json'
        }

        # Prepare messages array
        messages = []
        if self.system_prompt:
            messages.append({
                "role": "system",
                "content": self.system_prompt
            })

        messages.append({
            "role": "user",
            "content": prompt
        })

        # Use provided model or default to deepseek-chat for backward compatibility
        model_name = model if model is not None else "deepseek-chat"

        payload = {
            "model": model_name,
            "messages": messages,
            "temperature": self.temperature,
            "max_tokens": 4000,
            "stream": False
        }

        logger = logging.getLogger(__name__)
        logger.debug("OpenAIClient: Sending payload:")
        for message in messages:
            logger.debug("%s:\n%s", message['role'].upper(), message['content'])

        try:
            response = requests.post(self.svc_endpoint, headers=headers, json=payload)
            if response.status_code != 200:
                error_text = response.text()
                raise RuntimeError(f"OpenAI API request failed with status {response.status_code}: {error_text}")

            response_data = response.json()

            # Extract the response content from OpenAI API response format
            if 'choices' in response_data and len(response_data['choices']) > 0:
                choice = response_data['choices'][0]
                if 'message' in choice and 'content' in choice['message']:
                    content = choice['message']['content'].strip()
                    logger.debug("Executing python script: Received response content:\n%s", content)
                    self.execute_script(content)
                    return
                    
            raise RuntimeError(f"Unexpected OpenAI API response format: {response_data}")

        except requests.HTTPError as e:
            raise RuntimeError(f"HTTP client error when calling OpenAI API: {str(e)}") from e
        except json.JSONDecodeError as e:
            raise RuntimeError(f"Failed to parse OpenAI API response as JSON: {str(e)}") from e

In [ ]:
system_prompt="""Return only python script, no additional comments. Dont add 'python' on begginging of the script. Your response needs to work when executed by python: exec(). 
You can access global variables or create new ones, execute functions or methods using any variables from the context. 
Console outputs (e.g. showing print results) will be provided after each response starting with “>”. 
Your program works indefinitely and has unlimited memory. 
When you reach the goal or if you think the goal cannot be achieved end the program with exit(reason). 
You have access to the following tools:
> def clock() -> Clock
> def docs(class_name: type, method_name: str = None) -> str:
> def exit(reason: str)
> def send_message(message: str, attachment: Bytes, bool wait_for_reply = True) -> str | None

"""

In [ ]:
settings = AgentSettings()
temperature = 0.7
client = OpenAIClient(
    svc_endpoint=settings.chat_api_url,
    api_key=settings.chat_api_key,
    system_prompt=system_prompt,
    temperature=temperature
)

In [ ]:
result = client.chat("""You are a helpful assistant. Your goal is to answer the user’s question: “How many on-calls do I have in my schedule this month?”. 
Your Python console is ready with the initial context: 

print(user)
print(clock().now())
> first name: Adam, last name:  Kowalski
> 2026-01-08 12:43:11
""", model=settings.chat_api_model)